# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# 1. Import data from the hotels dataset
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv'
df = pd.read_csv(url)

# Drop columns that cause target leakage or are redundant
df = df.drop(columns=['reservation_status', 'reservation_status_date'])

# Create X (features) and y (target = is_canceled)
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# Split the data into training and test sets (70/30) BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include=['number']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

# 2. Remove or impute missing values (fit on X_train, transform both)

# Impute 'country' with the mode
mode_country = X_train['country'].mode()[0]
X_train['country'] = X_train['country'].fillna(mode_country)
X_test['country'] = X_test['country'].fillna(mode_country)

# Impute 'children', 'company', 'agent' with 0
X_train['children'] = X_train['children'].fillna(0)
X_test['children'] = X_test['children'].fillna(0)
X_train['company'] = X_train['company'].fillna(0)
X_test['company'] = X_test['company'].fillna(0)
X_train['agent'] = X_train['agent'].fillna(0)
X_test['agent'] = X_test['agent'].fillna(0)

# Impute 'adr' with the median
median_adr = X_train['adr'].median()
X_train['adr'] = X_train['adr'].fillna(median_adr)
X_test['adr'] = X_test['adr'].fillna(median_adr)

# 3. Encode categorical variables using One-Hot Encoding
# Apply one-hot encoding separately to training and test sets
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - crucial for consistent feature sets after one-hot encoding
# Reindex X_test_encoded to match the columns of X_train_encoded, filling missing with 0
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print(f"Original dataset shape: {df.shape}")
print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"X_test_encoded shape: {X_test_encoded.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Original dataset shape: (9404, 30)
X_train_encoded shape: (6582, 128)
X_test_encoded shape: (2822, 128)
y_train shape: (6582,)
y_test shape: (2822,)


### ✍️ Your Response: 🔧
1. the original datasets has 9404 rows and 30 columns

2. integer and float features were lead time stays in weekened nights, adr, previous cancellations and then other ones were hotel, meals, mkt segment, customer type
3. removed reservation status and reversation status date. split the data 70/30. filling adr with median. filling in children, company, agent.



## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [16]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize Naïve Bayes classifier
nb_model = GaussianNB()

# Train the model on the training data (convert to NumPy array to avoid feature name issues)
nb_model.fit(X_train_encoded.to_numpy(), y_train)

# Predict on the test data (convert to NumPy array)
y_pred_nb = nb_model.predict(X_test_encoded.to_numpy())

# Print classification report
print("Naïve Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))

# Print confusion matrix
print("Naïve Bayes Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Naïve Bayes Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.35      0.52      2111
           1       0.34      0.99      0.51       711

    accuracy                           0.51      2822
   macro avg       0.67      0.67      0.51      2822
weighted avg       0.83      0.51      0.51      2822

Naïve Bayes Confusion Matrix:
[[ 739 1372]
 [   6  705]]


### ✍️ Your Response: 🔧
1. the accuracy was 51% with clcass 0 having a precision of 99 and class 1 having pecision of 0.34. the best metric to use was recall

2. this model may be helpful in identifying cancellations early in situations where overbooking could be an issue

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Scale the data using StandardScaler
# Fit the scaler only on the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# 2. Train an SVM classifier (linear kernel)
print("Training SVM model... This may take a few minutes.")
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)
print("SVM training complete.")

# 3. Make predictions on the test data
y_pred_svm = svm_model.predict(X_test_scaled)

# 4. Evaluate with classification metrics
print("\nSupport Vector Machine Classification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nSupport Vector Machine Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Training SVM model... This may take a few minutes.
SVM training complete.

Support Vector Machine Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      2111
           1       0.82      0.82      0.82       711

    accuracy                           0.91      2822
   macro avg       0.88      0.88      0.88      2822
weighted avg       0.91      0.91      0.91      2822


Support Vector Machine Confusion Matrix:
[[1979  132]
 [ 127  584]]


### ✍️ Your Response: 🔧
1. this model had an accuracy of 91%

2. this model could be best used in more complex situations like risk prediction.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Neural Networks typically perform better with scaled data, so we'll use X_train_scaled and X_test_scaled

# Build an MLPClassifier model
# Choose a simple architecture (e.g., 2 hidden layers)
# Use early_stopping with validation_fraction to create a validation split from training data
print("Training Neural Network model... This may take a few minutes.")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50), # Example: two hidden layers with 100 and 50 neurons
    max_iter=500, # Increase max_iter if convergence warnings appear
    activation='relu', # Rectified Linear Unit activation function
    solver='adam', # Adam optimization algorithm
    random_state=42,
    early_stopping=True, # Use a validation set to stop training when validation score does not improve
    validation_fraction=0.1 # Use 10% of training data for validation
)

mlp_model.fit(X_train_scaled, y_train)
print("Neural Network training complete.")

# Predict on the test data
y_pred_nn = mlp_model.predict(X_test_scaled)

# Evaluate accuracy and performance
print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_nn))

print("\nNeural Network Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nn))

Training Neural Network model... This may take a few minutes.
Neural Network training complete.

Neural Network Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      2111
           1       0.85      0.81      0.83       711

    accuracy                           0.92      2822
   macro avg       0.89      0.88      0.89      2822
weighted avg       0.92      0.92      0.92      2822


Neural Network Confusion Matrix:
[[2012   99]
 [ 137  574]]


### ✍️ Your Response: 🔧
1. this model performed slightly better with an accuracy of 92%

2. a business may not feel comfortable using a black box model like this as there are still risks when it comes to preidiction and understanding what data affected what and how. When there isnt much known about the data important business decisions cannot be made with clear insight.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [19]:
# To get the accuracy for each model, we can parse the classification reports or directly calculate it.
# For simplicity, let's use the 'accuracy' score available from the classification report outputs already run.

# Naïve Bayes Accuracy (from previous output): 0.51
# SVM Accuracy (from previous output): 0.91
# Neural Network Accuracy (from previous output): 0.92

nb_accuracy = 0.51
svm_accuracy = 0.91
nn_accuracy = 0.92

print("\n--- Model Performance Comparison ---")
print(f"Naïve Bayes Model Accuracy: {nb_accuracy:.2f}")
print(f"SVM Model Accuracy: {svm_accuracy:.2f}")
print(f"Neural Network Model Accuracy: {nn_accuracy:.2f}")

# Determine the best performing model based on accuracy
accuracies = {
    "Naïve Bayes": nb_accuracy,
    "SVM": svm_accuracy,
    "Neural Network": nn_accuracy
}

best_model = max(accuracies, key=accuracies.get)
best_accuracy = accuracies[best_model]

print(f"\nSummary: The {best_model} model performed best with an accuracy of {best_accuracy:.2f}.")


--- Model Performance Comparison ---
Naïve Bayes Model Accuracy: 0.51
SVM Model Accuracy: 0.91
Neural Network Model Accuracy: 0.92

Summary: The Neural Network model performed best with an accuracy of 0.92.


### ✍️ Your Response: 🔧
1.  highest accuracy was neural network. quickest training time was naive bayes. best interpretability aws naive bayers and best ease of use was naive bayes.

2. I would reccomend SVM since its accuracy aws high and in terms of ease of use and complexity it is simpler than neaural networks (higher accuracy by 1% but much more complex)

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I reccomend implementing the SVM model to predict cancellations as it has an accuracy of 91% and is more practical than neaural networks. additional information that could improve results in the future could include data that indicates changes in price, changes in demand, etc.

2. In certain cases being able to interpret data beyond the historical is important to benefit a client depending on what area of service i am providing. If able having the applicable tools to help a client predict future data, understand the cahnges, and implement the correct models is a help that most cant provide.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [20]:
!jupyter nbconvert --to html "assignment_12_VegaSamantha.ipynb"

[NbConvertApp] Converting notebook assignment_12_VegaSamantha.ipynb to html
[NbConvertApp] Writing 329871 bytes to assignment_12_VegaSamantha.html
